# Constraint Validation: Catching Invalid Tool Parameters

Based on: [CCTU: Benchmark for Tool Use under Complex Constraints](https://arxiv.org/abs/2603.15309) (Mar 2026)

## The Problem

Your agent calls the right tool, but with wrong parameters:
- ❌ Date in the past: `search_flights(date="2020-01-01")`
- ❌ Check-out before check-in: `search_hotels(check_in="March 20", check_out="March 18")`
- ❌ Negative amount: `get_currency_exchange(amount=-100)`
- ❌ Missing required field: `book_hotel(hotel_name="", guest_name="")`

The CCTU paper found that **parameter constraint violations** are the most common failure mode, even more common than wrong tool selection.

## What We Compare

| Approach | What It Catches | Cost | When to Use |
|----------|----------------|:----:|------------|
| Deterministic constraints | Dates, ranges, required fields, formats | Free | Every tool call (production) |
| LLM `OutputEvaluator` | Semantic intent match | 1 call | Periodic quality checks |

We run a live agent, intercept tool calls with a **hook**, and validate parameters in real-time.

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Test the constraint checker on known-bad parameters

**What this does:** Runs the constraint checker on 7 pre-defined tool calls (2 valid, 5 invalid) to verify it catches known violations.

**Why we test with known inputs first:** Before connecting to a live agent, we validate that the constraint checker itself works correctly. This is the "unit test" step — if the checker misses known violations, it will also miss them in production.

**The constraint types being checked:**

| Constraint Type | What It Validates | Example Violation |
|----------------|-------------------|-------------------|
| **Date format** | Dates must be valid `YYYY-MM-DD` strings | `"bad-date"` is not a valid date |
| **Date range** | Dates must not be in the past | `"2020-01-01"` is in the past |
| **Date logic** | Check-out must be after check-in | Check-in March 20, check-out March 18 |
| **Required fields** | Certain string fields must be non-empty | `hotel_name=""` is empty |
| **Positive numbers** | Numeric amounts must be greater than zero | `amount=-100` is negative |

> **What to look for:** All 7 test cases should produce the expected result (valid or invalid). For each invalid call, the specific violation(s) should be listed. The checker accuracy should be 7/7 (100%). If any test fails, the constraint rules need adjustment before connecting to a live agent.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from constraint_checker import validate_tool_call

TEST_CALLS = [
    # Valid calls
    ("search_flights", {"origin": "NYC", "destination": "London", "date": "2026-04-15"}, True),
    ("get_currency_exchange", {"from_currency": "USD", "to_currency": "EUR", "amount": 500}, True),

    # Invalid calls
    ("search_flights", {"origin": "NYC", "destination": "London", "date": "2020-01-01"}, False),
    ("search_flights", {"origin": "", "destination": "London", "date": "bad-date"}, False),
    ("search_hotels", {"city": "Paris", "check_in": "2026-03-20", "check_out": "2026-03-18"}, False),
    ("book_hotel", {"hotel_name": "", "guest_name": "", "check_in": "2026-03-20", "check_out": "2026-03-22"}, False),
    ("get_currency_exchange", {"from_currency": "USD", "to_currency": "EUR", "amount": -100}, False),
]

print("=" * 70)
print("TEST 1: Constraint Checker on Known Inputs")
print("=" * 70)

correct = 0
for tool_name, params, expected_valid in TEST_CALLS:
    result = validate_tool_call(tool_name, params)
    matches = result["valid"] == expected_valid
    correct += matches
    icon = "✅" if matches else "❌"

    print(f"\n  {icon} {tool_name}({params})")
    print(f"     Expected: {'valid' if expected_valid else 'invalid'}, Got: {'valid' if result['valid'] else 'invalid'}")
    for v in result["violations"]:
        print(f"     ⚠️  {v}")

print(f"\n📊 Checker accuracy: {correct}/{len(TEST_CALLS)} ({correct/len(TEST_CALLS):.0%})")

---
## Step 2: Hook-based validation on a live agent

**What this does:** Creates a `ConstraintValidationHook` that intercepts every tool call's parameters *before* the tool executes, validates them against the constraint rules, and logs violations.

**Why use the hook pattern:** Without hooks, you would need to wrap every tool function with validation logic, or run validation after the fact. The hook pattern centralizes all validation in one place and runs automatically on every `BeforeToolCallEvent`. You add validation to an agent by passing `hooks=[validator]` — no changes to the tools or agent logic needed.

**Monitor mode vs block mode:** This demo uses monitor mode — violations are logged but the tool call still executes. In production, you can switch to block mode by setting `event.cancel_tool = "reason"` in the hook, which prevents the tool from executing and returns the cancellation reason to the agent.

> **What to look for:** For each query, watch which tool calls pass validation (checkmark) and which have violations (warning). Common violations from live agents include: dates the agent guessed rather than extracted from the query, or parameters the agent assumed were needed but the user did not specify. The summary at the end shows the total valid vs violated calls.

In [ ]:
import sys
sys.path.insert(0, "../01-tool-selection-accuracy")

from strands import Agent
from strands.models.openai import OpenAIModel
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent
from travel_tools import ALL_TOOLS

MODEL = "gpt-4o-mini"


class ConstraintValidationHook(HookProvider):
    """Validates tool parameters before execution. Logs violations."""

    def __init__(self):
        self.checks = []

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeToolCallEvent, self._validate)

    def _validate(self, event: BeforeToolCallEvent) -> None:
        tool_name = event.tool_use["name"]
        params = event.tool_use.get("input", {})
        result = validate_tool_call(tool_name, params)
        self.checks.append(result)

        if not result["valid"]:
            print(f"  ⚠️  CONSTRAINT VIOLATION: {tool_name}")
            for v in result["violations"]:
                print(f"     {v}")
        else:
            print(f"  ✅ {tool_name}({params}) — valid")


validator = ConstraintValidationHook()
agent = Agent(
    model=OpenAIModel(model_id=MODEL),
    tools=ALL_TOOLS,
    hooks=[validator],
    system_prompt="You are a travel assistant. Use tools to answer questions.",
)

print("=" * 70)
print("TEST 2: Live Agent with Constraint Validation Hook")
print("=" * 70)

QUERIES = [
    "Find flights from NYC to London for next Friday",
    "Search hotels in Paris for March 20-22",
    "Convert 500 USD to EUR",
]

for query in QUERIES:
    print(f"\n--- Query: '{query}' ---")
    agent(query)

# Summary
valid_count = sum(1 for c in validator.checks if c["valid"])
total = len(validator.checks)
print(f"\n📊 Constraint Validation Summary:")
print(f"   Total tool calls: {total}")
print(f"   Valid: {valid_count}")
print(f"   Violations: {total - valid_count}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor('white')

valid_count = sum(1 for c in validator.checks if c["valid"])
violated_count = len(validator.checks) - valid_count

labels = ['Valid', 'Violated']
values = [valid_count, violated_count]
colors = ['#4CAF50', '#E53935']

bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='white')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val}', ha='center', va='bottom', fontsize=16, fontweight='bold')

ax.set_ylabel('Number of Tool Calls', fontsize=12)
ax.set_title(f'Constraint Validation: Valid vs Violated Tool Calls\n(Total: {len(validator.checks)} tool calls)', fontweight='bold', fontsize=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Constraint validation is free and instant.** No LLM calls needed. Run it on every tool call in production.

2. **Hooks make it automatic.** The `ConstraintValidationHook` intercepts every `BeforeToolCallEvent` and validates parameters before the tool executes.

3. **Monitor mode vs. block mode.** This demo logs violations. In production, you can set `event.cancel_tool = True` to block invalid calls.

4. **Combine with semantic evaluation.** Constraints catch format/range errors. LLM evaluation catches intent mismatches. Use both.

| Check Type | Examples | Cost | Mode |
|-----------|---------|:----:|------|
| Date format | `YYYY-MM-DD` | Free | Block |
| Date range | Not in the past | Free | Block |
| Required fields | Non-empty strings | Free | Block |
| Positive numbers | `amount > 0` | Free | Block |
| Semantic intent | "Did the params match the query?" | 1 LLM call | Monitor |

**Next:** [Demo 03 - Parameter Correctness](../03-parameter-correctness/) — Semantic validation of whether parameters match user intent.